In [17]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.jd_parser import JDParser

In [18]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "candidates.jsonl"

from src.data_loader import CandidateDataLoader
from src.candidate_processor import CandidateProcessor
from src.semantic_matcher import SemanticMatcher
from src.jd_parser import JDParser
from src.ranking_engine import RankingEngine

In [19]:
parser = JDParser()

In [20]:
jd = """
We are looking for a Backend Engineer
with at least 3 years of experience.

Required Skills:
Python
SQL
AWS
Cloud

Preferred:
Docker
Kubernetes

Excellent communication and leadership skills.
"""

In [21]:
parsed_jd = parser.parse(jd)

parsed_jd

{'raw_text': 'We are looking for a Backend Engineer\nwith at least 3 years of experience.\n\nRequired Skills:\nPython\nSQL\nAWS\nCloud\n\nPreferred:\nDocker\nKubernetes\n\nExcellent communication and leadership skills.',
 'job_title': 'Backend Engineer',
 'minimum_experience': 3,
 'required_skills': ['Python', 'SQL', 'AWS', 'Docker', 'Kubernetes', 'Cloud'],
 'preferred_skills': [],
 'soft_skills': ['Leadership', 'Communication'],
 'keywords': ['Leadership',
  'Cloud',
  'SQL',
  'Kubernetes',
  'AWS',
  'Python',
  'Docker',
  'Communication']}

In [22]:
from src.ranking_engine import RankingEngine

In [23]:
ranker = RankingEngine()

In [24]:
from src.semantic_matcher import SemanticMatcher
from src.candidate_processor import CandidateProcessor
from src.data_loader import CandidateDataLoader

In [25]:
matcher = SemanticMatcher()
processor = CandidateProcessor()
loader = CandidateDataLoader(DATA_PATH)

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7360.43it/s]


Model loaded.


In [26]:
candidates = loader.load_candidates()

candidate = candidates[0]

profile = processor.process_profile(candidate)
skills = processor.process_skills(candidate)

candidate_text = " ".join([
    profile["headline"],
    profile["summary"],
    skills["skills_text"]
])

semantic_score = matcher.similarity(
    jd,
    candidate_text
)

final_score = ranker.rank(
    semantic_score,
    profile,
    skills,
    parsed_jd
)

print("Semantic Score :", semantic_score)
print("Final Score    :", final_score)

Semantic Score : 0.5945284366607666
Final Score    : 0.6612


In [27]:
scores = []

for candidate in candidates[:100]:

    profile = processor.process_profile(candidate)
    skills = processor.process_skills(candidate)

    candidate_text = " ".join([
        profile["headline"],
        profile["summary"],
        skills["skills_text"]
    ])

    semantic = matcher.similarity(
        jd,
        candidate_text
    )
    
    final = ranker.rank(
        semantic,
        profile,
        skills,
        parsed_jd
    )

    scores.append({
        "candidate_id": candidate["candidate_id"],
        "name": profile["name"],
        "semantic_score": round(semantic, 4),
        "final_score": final
    })

In [28]:
ranked = sorted(
    scores,
    key=lambda x: x["final_score"],
    reverse=True
)

ranked[:10]

[{'candidate_id': 'CAND_0000054',
  'name': 'Ved Malhotra',
  'semantic_score': 0.661,
  'final_score': 0.7444},
 {'candidate_id': 'CAND_0000072',
  'name': 'Diya Sharma',
  'semantic_score': 0.6742,
  'final_score': 0.7408},
 {'candidate_id': 'CAND_0000038',
  'name': 'Myra Trivedi',
  'semantic_score': 0.6708,
  'final_score': 0.7375},
 {'candidate_id': 'CAND_0000027',
  'name': 'Avni Pandey',
  'semantic_score': 0.6437,
  'final_score': 0.727},
 {'candidate_id': 'CAND_0000023',
  'name': 'Kavya Nair',
  'semantic_score': 0.6732,
  'final_score': 0.7232},
 {'candidate_id': 'CAND_0000067',
  'name': 'Shreya Bose',
  'semantic_score': 0.6677,
  'final_score': 0.7177},
 {'candidate_id': 'CAND_0000044',
  'name': 'Vihaan Naidu',
  'semantic_score': 0.6491,
  'final_score': 0.7158},
 {'candidate_id': 'CAND_0000051',
  'name': 'Meera Arora',
  'semantic_score': 0.6597,
  'final_score': 0.7097},
 {'candidate_id': 'CAND_0000091',
  'name': 'Aisha Desai',
  'semantic_score': 0.647,
  'final_s

In [29]:
for c in ranked[:10]:
    print(
        f"{c['candidate_id']} | "
        f"{c['name']} | "
        f"Semantic={c['semantic_score']} | "
        f"Final={c['final_score']}"
    )

CAND_0000054 | Ved Malhotra | Semantic=0.661 | Final=0.7444
CAND_0000072 | Diya Sharma | Semantic=0.6742 | Final=0.7408
CAND_0000038 | Myra Trivedi | Semantic=0.6708 | Final=0.7375
CAND_0000027 | Avni Pandey | Semantic=0.6437 | Final=0.727
CAND_0000023 | Kavya Nair | Semantic=0.6732 | Final=0.7232
CAND_0000067 | Shreya Bose | Semantic=0.6677 | Final=0.7177
CAND_0000044 | Vihaan Naidu | Semantic=0.6491 | Final=0.7158
CAND_0000051 | Meera Arora | Semantic=0.6597 | Final=0.7097
CAND_0000091 | Aisha Desai | Semantic=0.647 | Final=0.697
CAND_0000043 | Aarav Sen | Semantic=0.6445 | Final=0.6945
